In [1]:
import socket
import json
import threading

In [ ]:
# 1. 서버 기본 설정

HOST = '192.168.0.49' # 서버의 IP 주소
PORT = 9887 # (0~ 65535 중 하나 선택) 
MAX_CLIENTS = 10


# -----------------------------------
# 2. 분석 함수 정의
# -----------------------------------
def analyze_text(request):
    """
    클라이언트의 요청(JSON)을 받아 분석 결과를 반환하는 함수
    request: dict (예: {"mode": "sentiment", "text": "문장"})
    """
    mode = request.get("mode")  # 분석 모드
    text = request.get("text", "")

    # (1) 문자열 길이 분석
    if mode == "length":
        return {
            "result": len(text),
            "desc": f"문자 길이는 {len(text)}자입니다."
        }

    # (2) 간단한 감성 분석 (규칙 기반)
    elif mode == "sentiment":
        if any(word in text for word in ["좋아", "행복", "멋져", "훌륭"]):
            sentiment = "positive"
        elif any(word in text for word in ["싫어", "불만", "짜증", "나빠"]):
            sentiment = "negative"
        else:
            sentiment = "neutral"
        return {
            "result": sentiment,
            "desc": f"감정 분석 결과: {sentiment}"
        }

    # (3) 키워드 탐지
    elif mode == "keyword":
        keywords = ["고장", "위치", "긁힘", "불량", "찍힘"]
        found = [k for k in keywords if k in text]
        return {
            "result": found,
            "desc": f"키워드 발견: {', '.join(found) if found else '없음'}"
        }

    # (4) 지원하지 않는 모드 처리
    else:
        return {"error": f"지원하지 않는 분석 모드입니다: {mode}"}



# 3. 클라이언트 처리 스레드 함수
def handle_client(client_socket, address):
    """
    각 클라이언트 연결마다 실행되는 함수
    : param client_socket : 해당 클라이언트의 소켓
    : parma address : 해당 클라이언트의 주소 정보
    """

    print(f"클라이언트 {address} 연결됨")

    while True:
        try:
            data = client_socket.recv(2048).decode() #클라이언트로부터 데이터 수신

            if not data:
                print(f" {address} 연결 끊김")
                break

            if data.lower() == "exit":
                print(f"{address} 종료 수신")
                break

            # json data parsing
            try:
                request = json.loads(data)
                result = analyze_text(request)
            except json.JSONDecodeError:
                result = {"error":"잘못된 json 형식입니다"}

            # 응답 전송(json -> bytes)
            response = json.dumps(result, ensure_ascii=False)
            client_socket.sendall(response.encode())

        except ConnectionResetError:
            print(f" {address} 비정상 종료")
            break

    client_socket.close()
    print(f"클라이언트 {address} 세션 종료 완료")


# 4. 서버 메인 실행부

def start_server():
    """
    메인 서버 함수 : 클라이언트 접속을 대기하고, 접속 시마다
    새로운 쓰레드 생성하여 handle_client함수 실행
    """
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_socket.bind((HOST,PORT))
    server_socket.listen(MAX_CLIENTS)

    print(f" AI 서버 실행 중 ....{HOST}:{PORT}\n")
    print(f" 최대 {MAX_CLIENTS}개의 클라이언트 동시 접속 가능\n")

    try:
        while True:
            client_socket, addr = server_socket.accept()
            client_thread = threading.Thread(
                target = handle_client, args=(client_socket, addr), daemon = True
            )
            client_thread.start()
    except KeyboardInterrupt:
        print("\n 서버 수동 종료 감지")
    finally :
        server_socket.close()
        print(" 서버 완전 종료")

# 5. 실행 시작

if __name__ == "__main__":
    start_server()




                

 AI 서버 실행 중 ....192.168.0.49:9887

 최대 10개의 클라이언트 동시 접속 가능

클라이언트 ('192.168.0.49', 61225) 연결됨
('192.168.0.49', 61225) 종료 수신
클라이언트 ('192.168.0.49', 61225) 세션 종료 완료
클라이언트 ('192.168.0.49', 61411) 연결됨
 ('192.168.0.49', 61411) 연결 끊김
클라이언트 ('192.168.0.49', 61411) 세션 종료 완료
